# Per-90 Performance Stats from Events

Phase 1 — Data Inspection, part 7. This notebook turns raw events into per-player-per-season performance stats (totals plus per-90 rates) on the joined rows of player_season_value.csv, and saves the modeling table player_season_features.csv.

Design notes: the season list (23 league-seasons) comes from the join file itself — cups produced zero matched rows, so there is nothing to build for them. Minutes come from lineup time spans (from/to); substitutes who never entered count zero. Every event is linked to a lineup player in the same match, with the link rate checked. Assists are out of scope for v1 (no direct assist field in the events).

## Cell 1 — Setup and season list from the join table

Reads player_season_value.csv (the handover's join deliverable) and maps each league_season label back to its match files: leagues split on the last space and resolve via competitions.json; the five cups map explicitly (absent from the file, kept for completeness). Prints match counts per season — the denominators for everything after.

In [1]:
import json, os, re, sys
import pandas as pd
import numpy as np

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
from src.join import normalize_name

TM_DIR = os.path.join(ROOT, 'data', 'transfermarkt')
SB_DATA_DIR = os.path.join(ROOT, 'data', 'statsbomb', 'data')
OUT_DIR = os.path.join(ROOT, 'data', 'processed')

JOIN = pd.read_csv(os.path.join(OUT_DIR, 'player_season_value.csv'))
print('join rows:', len(JOIN), '| seasons:', JOIN['league_season'].nunique())

TOURN_MAP = {'Euro 2020': (55, 43), 'Euro 2024': (55, 282), 'World Cup 2018': (43, 3),
             'World Cup 2022': (43, 106), 'Copa America 2024': (223, 282)}
sb_comp = pd.read_json(os.path.join(SB_DATA_DIR, 'competitions.json'))
sb_comp['label'] = sb_comp['competition_name'] + ' ' + sb_comp['season_name']
label2file = dict(zip(sb_comp['label'], zip(sb_comp['competition_id'], sb_comp['season_id'])))

SEASONS = []
for label in sorted(JOIN['league_season'].unique()):
    if label in TOURN_MAP:
        comp, season = TOURN_MAP[label]
    else:
        comp, season = label2file[label]
    fp = os.path.join(SB_DATA_DIR, 'matches', str(comp), f'{season}.json')
    mids = [m['match_id'] for m in json.load(open(fp, encoding='utf-8'))]
    SEASONS.append((label, comp, season, mids))
print('seasons:', len(SEASONS), '| total matches:', sum(len(m[3]) for m in SEASONS))
print(pd.DataFrame([(l, len(m)) for l, _, _, m in SEASONS], columns=['season', 'matches']).to_string())


join rows: 6454 | seasons: 23
seasons: 23 | total matches: 1923
                        season  matches
0      1. Bundesliga 2015/2016       34
1      1. Bundesliga 2023/2024       34
2   Champions League 2012/2013        1
3   Champions League 2013/2014        1
4   Champions League 2014/2015        1
5   Champions League 2015/2016        1
6   Champions League 2016/2017        1
7   Champions League 2017/2018        1
8   Champions League 2018/2019        1
9            La Liga 2012/2013       32
10           La Liga 2013/2014       31
11           La Liga 2014/2015       38
12           La Liga 2015/2016      380
13           La Liga 2016/2017       34
14           La Liga 2017/2018       36
15           La Liga 2018/2019       34
16           La Liga 2019/2020       33
17           La Liga 2020/2021       35
18           Ligue 1 2015/2016      377
19           Ligue 1 2021/2022       26
20           Ligue 1 2022/2023       32
21    Premier League 2015/2016      380
22           Ser

## Cell 2 — Minutes and positions from lineups

Turns each time span (from/to, minutes:seconds) into minutes played. Open-ended spans split two ways: starting at 00:00 means an unused substitute (zero minutes); starting later means a substitute who played to the final whistle (90:00 minus entry time — approximate, stated). Also records who started and the opening position name for the season's main position.

In [2]:
def to_min(t):
    if t is None:
        return None
    m, s = t.split(':')
    return int(m) + int(s) / 60

min_rows = []
for label, comp, season, mids in SEASONS:
    for mid in mids:
        fp = os.path.join(SB_DATA_DIR, 'lineups', f'{mid}.json')
        for t in json.load(open(fp, encoding='utf-8')):
            for p in t['lineup']:
                mins, first_pos, started = 0.0, None, False
                for i, pos in enumerate(p['positions']):
                    f, e = to_min(pos.get('from')), to_min(pos.get('to'))
                    if i == 0:
                        first_pos = pos.get('position')
                        started = (pos.get('from') == '00:00')
                    if e is None:
                        mins += 0.0 if pos.get('from') == '00:00' else max(0.0, 90.0 - (f or 0.0))
                    else:
                        mins += max(0.0, e - (f or 0.0))
                min_rows.append((label, mid, p['player_name'], t['team_name'],
                                 round(mins, 2), started and mins > 0, first_pos))
MIN = pd.DataFrame(min_rows, columns=['league_season', 'match_id', 'sb_player', 'sb_team',
                                       'minutes', 'started', 'position'])
print('rows:', len(MIN))
print('zero-minute rows (unused subs):', (MIN['minutes'] == 0).sum())
print('minutes distribution:')
print(MIN[MIN['minutes'] > 0]['minutes'].describe().to_string())


rows: 72464
zero-minute rows (unused subs): 42498
minutes distribution:
count    29966.000000
mean        56.378406
std         30.764537
min          0.020000
25%         25.120000
50%         65.075000
75%         89.130000
max        143.020000


## Cell 3 — Events engine (counters per player per match)

One pass over the season's event files. Counters follow the checked outcome words: passes with no outcome are complete; shots count goals on outcome Goal; dribbles split complete/incomplete; duels split won (Success In Play, Won) vs lost. Events are keyed by cleaned player name and linked to the same match's lineups; the unlinked share (events naming no lineup player) is the quality metric — near zero is expected since both come from the same feed.

In [3]:
WON_DUEL = {'Success In Play', 'Won'}

def count_match(mid):
    ev = json.load(open(os.path.join(SB_DATA_DIR, 'events', f'{mid}.json'), encoding='utf-8'))
    agg, unattributed = {}, 0

    def row(pl):
        return agg.setdefault(pl, {'shots': 0, 'goals': 0, 'xg': 0.0, 'passes': 0,
            'pass_ok': 0, 'drib_att': 0, 'drib_ok': 0, 'carries': 0, 'pressures': 0,
            'duels_w': 0, 'duels_l': 0, 'interceptions': 0, 'recoveries': 0,
            'blocks': 0, 'clearances': 0, 'fouls_w': 0, 'fouls_c': 0})

    for e in ev:
        t = e['type']['name']
        pl = (e.get('player') or {}).get('name')
        if pl is None:
            continue
        r = row(normalize_name(pl))
        if t == 'Shot' and isinstance(e.get('shot'), dict):
            r['shots'] += 1
            r['xg'] += e['shot'].get('statsbomb_xg') or 0.0
            if e['shot'].get('outcome', {}).get('name') == 'Goal':
                r['goals'] += 1
        elif t == 'Pass' and isinstance(e.get('pass'), dict):
            r['passes'] += 1
            if e['pass'].get('outcome') is None:
                r['pass_ok'] += 1
        elif t == 'Dribble' and isinstance(e.get('dribble'), dict):
            r['drib_att'] += 1
            if e['dribble'].get('outcome', {}).get('name') == 'Complete':
                r['drib_ok'] += 1
        elif t == 'Carry':
            r['carries'] += 1
        elif t == 'Duel' and isinstance(e.get('duel'), dict):
            out = e['duel'].get('outcome', {}).get('name')
            if out in WON_DUEL:
                r['duels_w'] += 1
            elif out is not None:
                r['duels_l'] += 1
        elif t == 'Interception':
            r['interceptions'] += 1
        elif t == 'Ball Recovery':
            r['recoveries'] += 1
        elif t == 'Block':
            r['blocks'] += 1
        elif t == 'Clearance':
            r['clearances'] += 1
        elif t == 'Foul Won':
            r['fouls_w'] += 1
        elif t == 'Foul Committed':
            r['fouls_c'] += 1
        elif t == 'Pressure':
            r['pressures'] += 1
    return agg

ev_rows = []
lineup_keys = set(zip(MIN['match_id'], MIN['sb_player'].apply(normalize_name)))
unatt, total_pl_ev = 0, 0
for label, comp, season, mids in SEASONS:
    for mid in mids:
        for key, r in count_match(mid).items():
            total_pl_ev += 1
            if (mid, key) not in lineup_keys:
                unatt += 1
                continue
            ev_rows.append((label, mid, key, *r.values()))
EV = pd.DataFrame(ev_rows, columns=['league_season', 'match_id', 'key', 'shots', 'goals', 'xg',
    'passes', 'pass_ok', 'drib_att', 'drib_ok', 'carries', 'pressures', 'duels_w', 'duels_l',
    'interceptions', 'recoveries', 'blocks', 'clearances', 'fouls_w', 'fouls_c'])
print('attributed rows:', len(EV), '| unattributed player-keys:', unatt, f'({100 * unatt / max(total_pl_ev, 1):.2f}%)')
print(EV[['shots', 'goals', 'xg', 'passes']].sum().to_string())


attributed rows: 53685 | unattributed player-keys: 0 (0.00%)
shots     4.820100e+04
goals     5.222000e+03
xg        4.937183e+03
passes    1.923224e+06


Counter list: shots/goals/xG, passes and completion, dribbles, carries, pressures, duels won/lost, interceptions, recoveries, blocks, clearances, fouls won/committed. Pressing volume comes from Pressure events, which carry player names in this feed. Assists stay out of v1 (no direct assist field); a follow-up can derive them from shot-assist links.

## Cell 4 — Season totals, per-90 rates, and the modeling table

Match rows join minutes on (match, cleaned name), sum to player-season, convert totals to per-90 rates over the minute total, take the most common main position, and merge market prices from the join table. Saves player_season_features.csv — the table the price model trains on.

In [4]:
MIN['key'] = MIN['sb_player'].apply(normalize_name)
M = MIN.merge(EV, on=['league_season', 'match_id', 'key'], how='left')
for c in ['shots', 'goals', 'xg', 'passes', 'pass_ok', 'drib_att', 'drib_ok', 'carries',
          'pressures', 'duels_w', 'duels_l', 'interceptions', 'recoveries', 'blocks',
          'clearances', 'fouls_w', 'fouls_c']:
    M[c] = M[c].fillna(0)
def first_mode(s):
    m = s.dropna().mode()
    return m.iloc[0] if len(m) else None

g = M.groupby(['league_season', 'sb_player', 'sb_team'])
F = g.agg(minutes=('minutes', 'sum'), matches=('minutes', lambda s: (s > 0).sum()),
          starts=('started', 'sum'), position=('position', first_mode),
          shots=('shots', 'sum'), goals=('goals', 'sum'), xg=('xg', 'sum'),
          passes=('passes', 'sum'), pass_ok=('pass_ok', 'sum'),
          drib_att=('drib_att', 'sum'), drib_ok=('drib_ok', 'sum'), carries=('carries', 'sum'),
          pressures=('pressures', 'sum'), duels_w=('duels_w', 'sum'), duels_l=('duels_l', 'sum'),
          interceptions=('interceptions', 'sum'), recoveries=('recoveries', 'sum'),
          blocks=('blocks', 'sum'), clearances=('clearances', 'sum'),
          fouls_w=('fouls_w', 'sum'), fouls_c=('fouls_c', 'sum')).reset_index()
p90 = F['minutes'].clip(lower=1) / 90
for c in ['shots', 'goals', 'xg', 'passes', 'drib_att', 'drib_ok', 'carries', 'pressures',
          'duels_w', 'interceptions', 'recoveries', 'blocks', 'clearances']:
    F[c + '_p90'] = (F[c] / p90).round(3)
F['pass_pct'] = (100 * F['pass_ok'] / F['passes'].clip(lower=1)).round(1)
F = F.merge(JOIN[['league_season', 'sb_player', 'sb_team', 'player_id', 'player_name', 'tm_club',
                  'method', 'date', 'market_value_in_eur']],
            on=['league_season', 'sb_player', 'sb_team'], how='inner')
assert F.duplicated(['league_season', 'sb_player', 'sb_team']).sum() == 0, 'cartesian merge!'
os.makedirs(OUT_DIR, exist_ok=True)
F.to_csv(os.path.join(OUT_DIR, 'player_season_features.csv'), index=False)
print('feature rows:', len(F), '| columns:', len(F.columns))
print(F.head(8).to_string())


feature rows: 6454 | columns: 44
             league_season                sb_player              sb_team  minutes  matches  starts                   position  shots  goals        xg  passes  pass_ok  drib_att  drib_ok  carries  pressures  duels_w  duels_l  interceptions  recoveries  blocks  clearances  fouls_w  fouls_c  shots_p90  goals_p90  xg_p90  passes_p90  drib_att_p90  drib_ok_p90  carries_p90  pressures_p90  duels_w_p90  interceptions_p90  recoveries_p90  blocks_p90  clearances_p90  pass_pct  player_id           player_name              tm_club method        date  market_value_in_eur
0  1. Bundesliga 2015/2016               Aaron Hunt         Hamburger SV    81.97        1       1  Center Attacking Midfield    5.0    0.0  0.294081    82.0     49.0       5.0      4.0     56.0       31.0      3.0      4.0            0.0         9.0     2.0         0.0      8.0      4.0      5.490       0.00   0.323      90.033         5.490        4.392       61.486         34.037        3.294   

## Cell 5 — Sanity checks

Three independent sniffs: the minutes spread must look like football (a bulge at full-match values, a tail of substitutes, zeros for unused bench); the xG/90 leaders among high-minute attackers must be recognizable names; goals must track xG positively. A failure in any of these sends the investigation back to Cells 2–3, not forward to modeling.

In [5]:
played = F[F['minutes'] > 0]
print('minute bands:')
print(pd.cut(played['minutes'], bins=[0, 500, 1500, 2500, 3500, 6000]).value_counts().sort_index().to_string())
print()
att = played[(played['minutes'] >= 1000) & (played['position'].str.contains('Forward|Wing|Striker', na=False))]
print('top xG/90 (min 1000, forwards):')
print(att.sort_values('xg_p90', ascending=False)[['league_season', 'sb_player', 'sb_team', 'minutes', 'goals', 'xg_p90']].head(10).to_string())
print()
print('goals vs xG correlation:', round(played[['goals', 'xg']].corr().iloc[0, 1], 3))
print('nulls:', F.isna().sum()[F.isna().sum() > 0].to_dict() or 'NONE')


minute bands:
minutes
(0, 500]        3407
(500, 1500]      796
(1500, 2500]     180
(2500, 3500]       1
(3500, 6000]       0

top xG/90 (min 1000, forwards):
                 league_season                            sb_player              sb_team  minutes  goals  xg_p90
1805         La Liga 2014/2015       Lionel Andrés Messi Cuccittini            Barcelona  1035.00   43.0   2.532
5224         Ligue 1 2022/2023                 Kylian Mbappé Lottin  Paris Saint-Germain  1023.43   27.0   2.049
2015         La Liga 2015/2016  Cristiano Ronaldo dos Santos Aveiro          Real Madrid  1561.21   35.0   1.843
5630  Premier League 2015/2016                          Jamie Vardy       Leicester City  1099.53   24.0   1.825
3297         La Liga 2018/2019       Lionel Andrés Messi Cuccittini            Barcelona  1173.65   36.0   1.601
6169         Serie A 2015/2016              Gonzalo Gerardo Higuaín               Napoli  1541.64   36.0   1.461
2179         La Liga 2015/2016                   

## Next steps

1. Review Cell 5: if names, bands, and correlation look like football, the table stands.
2. Design Postgres from player_season_features.csv and load it (handover data layer).
3. Study price relationships, then compare models from baseline to XGBoost.